# Basic CNN

Classify MRI crops as Normal/Mild, Moderate or Severe.

## Install dependencies

In [ ]:
%pip install -q pydicom python-gdcm

## Import libraries

In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import pydicom
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, log_loss, precision_recall_fscore_support
)

## Set the configuration

Use the same seed, study subset and split file in all three notebooks. Change `RUN_ROOT` for a new run.

In [ ]:
SEED = 42
EPOCHS = 20
MAX_STUDIES = 0  # Use all studies; set 20 for a quick check.
CROP_SIZE = 128
IMAGE_SIZE = 128
BATCH_SIZE = 16
LEARNING_RATE = 0.001
WORKERS = 2
MODEL_NAME = "baseline"
RUN_ROOT = Path("/kaggle/working/cnn_densenet_full_v1")
SPLIT_FILE = RUN_ROOT / "split.csv"
output = RUN_ROOT / MODEL_NAME

input_dir = Path("/kaggle/input")
paths = list(input_dir.glob("*/train_label_coordinates.csv"))
paths += list(input_dir.glob("*/*/train_label_coordinates.csv"))
assert len(paths) == 1, "Attach exactly one RSNA dataset."
DATA_ROOT = paths[0].parent
device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Enable a GPU in Kaggle settings before training."

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print("Device:", device, "| Data:", DATA_ROOT)

## Match coordinates with labels

In [ ]:
CLASS_NAMES = ["Normal/Mild", "Moderate", "Severe"]
labels = pd.read_csv(DATA_ROOT / "train.csv").melt(
    id_vars="study_id", var_name="target", value_name="severity"
)
coords = pd.read_csv(DATA_ROOT / "train_label_coordinates.csv")
coords["target"] = (
    coords.condition.str.lower().str.replace(" ", "_", regex=False)
    + "_" + coords.level.str.lower().str.replace("/", "_", regex=False)
)
samples = coords.merge(labels, on=["study_id", "target"], validate="many_to_one")
samples = samples.dropna(subset=["severity"]).copy()
samples["label"] = samples.severity.map(dict(zip(CLASS_NAMES, range(3))))
assert samples.label.notna().all(), "Unknown severity label."
samples["label"] = samples.label.astype(int)
studies = sorted(samples.study_id.unique())
if MAX_STUDIES:
    studies = sorted(random.Random(SEED).sample(studies, min(MAX_STUDIES, len(studies))))
    samples = samples[samples.study_id.isin(studies)]
assert len(studies) >= 4, "At least four studies are needed."

## Split the data

Split by study: 70% training, 15% validation, 15% test. All crops from one study stay together.

In [ ]:
if SPLIT_FILE.exists():
    split = pd.read_csv(SPLIT_FILE)
    assert set(split.columns) == {"study_id", "split"}
    assert not split.study_id.duplicated().any()
    assert set(split.study_id) == set(studies), "Split does not match the selected studies."
    assert set(split.split) == {"train", "val", "test"}
else:
    train_ids, remaining = train_test_split(studies, test_size=0.3, random_state=SEED)
    val_ids, test_ids = train_test_split(remaining, test_size=0.5, random_state=SEED)
    split = pd.DataFrame({"study_id": studies, "split": "train"})
    split.loc[split.study_id.isin(val_ids), "split"] = "val"
    split.loc[split.study_id.isin(test_ids), "split"] = "test"
    SPLIT_FILE.parent.mkdir(parents=True, exist_ok=True)
    split.to_csv(SPLIT_FILE, index=False)

samples = samples.merge(split, on="study_id", validate="many_to_one")
samples["path"] = (
    "train_images/" + samples.study_id.astype(str) + "/"
    + samples.series_id.astype(str) + "/" + samples.instance_number.astype(str) + ".dcm"
)
display(samples.groupby("split").agg(studies=("study_id", "nunique"), crops=("label", "size")))

## Read and crop MRI images

In [ ]:
class SpineCrops(Dataset):
    def __init__(self, rows):
        self.rows = rows.to_dict("records")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        dicom = pydicom.dcmread(DATA_ROOT / row["path"])
        pixels = torch.tensor(dicom.pixel_array.astype(np.float32))
        pixels = pixels * float(dicom.get("RescaleSlope", 1)) + float(dicom.get("RescaleIntercept", 0))
        pixels = (pixels - pixels.min()) / (pixels.max() - pixels.min()).clamp_min(1e-6)
        if dicom.PhotometricInterpretation == "MONOCHROME1":
            pixels = 1 - pixels

        height, width = pixels.shape
        x, y = row["x"], row["y"]
        assert 0 <= x < width and 0 <= y < height, "Crop center is outside the image."
        x, y = min(round(x), width - 1), min(round(y), height - 1)
        # Padding keeps edge crops centered and the same size.
        half = CROP_SIZE // 2
        pixels = F.pad(pixels, (half, half, half, half))
        crop = pixels[y:y + CROP_SIZE, x:x + CROP_SIZE]
        image = F.interpolate(crop[None, None], size=(IMAGE_SIZE, IMAGE_SIZE),
                              mode="bilinear", align_corners=False)[0]
        return image, row["label"]

## Create data loaders

In [ ]:
train_rows = samples[samples.split == "train"]
val_rows = samples[samples.split == "val"]
train_data = SpineCrops(train_rows)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=WORKERS, generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(SpineCrops(val_rows), batch_size=BATCH_SIZE, num_workers=WORKERS)

image, label = train_data[0]
plt.imshow(image[0], cmap="gray", vmin=0, vmax=1)
plt.title(CLASS_NAMES[label])
plt.axis("off")
plt.show()

## Build the model

In [ ]:
def build_baseline():
    # Three Conv-BN-ReLU-Pool blocks.
    return nn.Sequential(
        nn.Conv2d(1, 16, 3, padding=1),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(64, 3),
    )

## Define evaluation metrics

In [ ]:
def evaluate(model, loader):
    model.eval()
    probabilities, targets = [], []
    with torch.inference_mode():
        for images, labels in loader:
            probabilities.append(model(images.to(device)).softmax(1).cpu().numpy())
            targets.extend(labels.tolist())
    probabilities = np.concatenate(probabilities).astype(np.float64)
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    predictions = probabilities.argmax(1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        targets, predictions, labels=[0, 1, 2], average="macro", zero_division=0
    )
    metrics = {
        "log_loss": log_loss(targets, probabilities, labels=[0, 1, 2]),
        "accuracy": accuracy_score(targets, predictions),
        "macro_precision": precision, "macro_recall": recall, "macro_f1": f1,
        "confusion_matrix": confusion_matrix(targets, predictions, labels=[0, 1, 2]).tolist(),
        "class_counts": np.bincount(targets, minlength=3).tolist(), "crops": len(targets),
    }
    return metrics, probabilities


def save_results(metrics, probabilities, rows, split_name):
    result = dict(metrics, split=split_name)
    (output / f"{split_name}_metrics.json").write_text(json.dumps(result, indent=2))
    predictions = rows.copy()
    predictions[["normal_mild", "moderate", "severe"]] = probabilities
    predictions.to_csv(output / f"{split_name}_predictions.csv", index=False)

## Train the model

Save the checkpoint with the lowest validation log loss. To evaluate an existing run, skip this cell after running the setup cells above.

In [ ]:
output.mkdir(parents=True, exist_ok=True)
assert not any(output.iterdir()), "Results already exist. Change RUN_ROOT, or skip training to evaluate."
model = build_baseline().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
config = {
    "pipeline_version": 2, "model": MODEL_NAME, "pretrained": False,
    "seed": SEED, "epochs": EPOCHS, "max_studies": MAX_STUDIES,
    "crop_size": CROP_SIZE, "image_size": IMAGE_SIZE, "batch_size": BATCH_SIZE,
    "lr": LEARNING_RATE, "torch_version": str(torch.__version__),
    "parameters": sum(p.numel() for p in model.parameters()),
}
(output / "config.json").write_text(json.dumps(config, indent=2))
samples.to_csv(output / "samples.csv", index=False)
history, best_loss = [], float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for step, (images, labels) in enumerate(train_loader, 1):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        if step % 100 == 0:
            print(f"Epoch {epoch}: batch {step}/{len(train_loader)}", flush=True)

    metrics, probabilities = evaluate(model, val_loader)
    metrics["epoch"] = epoch
    history.append({"epoch": epoch, "train_loss": total_loss / len(train_data),
                    "val_loss": metrics["log_loss"], "accuracy": metrics["accuracy"],
                    "macro_f1": metrics["macro_f1"]})
    pd.DataFrame(history).to_csv(output / "history.csv", index=False)
    print(history[-1], flush=True)
    if metrics["log_loss"] < best_loss:
        best_loss = metrics["log_loss"]
        torch.save({"model": model.state_dict(), "config": config, "epoch": epoch}, output / "best.pt")
        save_results(metrics, probabilities, val_rows, "val")

## Evaluate on the test set

Run after fixing the training configuration. Test scores must not be used to choose epochs or hyperparameters. These metrics measure classification of crops with supplied coordinates.

In [ ]:
saved_samples = pd.read_csv(output / "samples.csv")
pd.testing.assert_frame_equal(saved_samples, samples.reset_index(drop=True), check_dtype=False)
checkpoint = torch.load(output / "best.pt", map_location="cpu", weights_only=True)
config = checkpoint["config"]
assert config["model"] == MODEL_NAME
assert config["crop_size"] == CROP_SIZE and config["image_size"] == IMAGE_SIZE
model = build_baseline().to(device)
model.load_state_dict(checkpoint["model"])

test_rows = samples[samples.split == "test"]
test_loader = DataLoader(SpineCrops(test_rows), batch_size=BATCH_SIZE, num_workers=WORKERS)
metrics, probabilities = evaluate(model, test_loader)
metrics["epoch"] = checkpoint["epoch"]
save_results(metrics, probabilities, test_rows, "test")
comparison = pd.DataFrame([{
    "model": MODEL_NAME, "pretrained": config["pretrained"], "epoch": checkpoint["epoch"],
    "test_log_loss": metrics["log_loss"], "test_accuracy": metrics["accuracy"],
    "test_macro_precision": metrics["macro_precision"],
    "test_macro_recall": metrics["macro_recall"], "test_macro_f1": metrics["macro_f1"],
}])
comparison.to_csv(output / "comparison.csv", index=False)
display(comparison)
print("Results:", output)

## View the results

In [ ]:
history = pd.read_csv(output / "history.csv")
history.plot(x="epoch", y=["train_loss", "val_loss"], ylabel="Loss")
plt.show()

matrix = pd.DataFrame(metrics["confusion_matrix"], index=CLASS_NAMES, columns=CLASS_NAMES)
display(matrix.rename_axis("True label").rename_axis("Predicted label", axis=1))